In [1]:
import pandas as pd

# Wczytanie surowych danych
flights = pd.read_csv("data/flights.csv")
airports = pd.read_csv("data/airports.csv")
airlines = pd.read_csv("data/airlines.csv")
planes = pd.read_csv("data/planes.csv")
weather = pd.read_csv("data/weather.csv")

print("Surowe dane wczytane pomyślnie.")

Surowe dane wczytane pomyślnie.


## Krok 2: Analiza i usunięcie duplikatów w tabeli `weather`


# Podgladanie duplikatow

In [2]:
klucze_pogody = ['origin', 'year', 'month', 'day', 'hour']
maska_duplikatow = weather.duplicated(subset=klucze_pogody, keep=False)
duplikaty_wiersze = weather[maska_duplikatow].sort_values(by=klucze_pogody)
print(" --- ZNALEZIONE DUPLIKATY W TABELI WEATHER ---")
display(duplikaty_wiersze[['origin', 'month', 'day', 'hour', 'temp', 'wind_speed', 'precip']])

 --- ZNALEZIONE DUPLIKATY W TABELI WEATHER ---


,origin,month,day,hour,temp,wind_speed,precip
7318,EWR,11,3,1,51.98,6.90468,0.0
7319,EWR,11,3,1,50.00,5.75390,0.0
16023,JFK,11,3,1,53.96,9.20624,0.0
16024,JFK,11,3,1,51.98,6.90468,0.0
24729,LGA,11,3,1,55.04,9.20624,0.0
24730,LGA,11,3,1,53.96,8.05546,0.0


# Usuwanie duplikatów

In [3]:
przed_czyszczeniem = len(weather)
weather_clean = weather.drop_duplicates(subset=klucze_pogody, keep='first').copy()
po_czyszczeniu = len(weather_clean)

print(f"\nUsunięto {przed_czyszczeniem - po_czyszczeniu} zduplikowane wiersze. Klucz główny pogody jest teraz w pełni unikalny!")


Usunięto 3 zduplikowane wiersze. Klucz główny pogody jest teraz w pełni unikalny!


## Krok 3: Standaryzacja (Usuwanie białych znaków)

In [4]:
# Usuwanie ukrytych spacji z kodów linii lotniczych
flights['carrier'] = flights['carrier'].astype(str).str.strip()
airlines['carrier'] = airlines['carrier'].astype(str).str.strip()

# Usuwanie ukrytych spacji z numerów rejestracyjnych samolotów
flights['tailnum'] = flights['tailnum'].astype(str).str.strip()
planes['tailnum'] = planes['tailnum'].astype(str).str.strip()

print("Klucze tekstowe (carrier, tailnum) zostały wyczyszczone z ukrytych białych znaków.")

Klucze tekstowe (carrier, tailnum) zostały wyczyszczone z ukrytych białych znaków.


## Krok 4: Wyodrębnienie lotów odwołanych i czyszczenie braków danych (`NaN`)

Wstępna analiza wykazała ponad 8000 braków danych w kolumnach `dep_delay` (opóźnienie wylotu) oraz `arr_delay` (opóźnienie przylotu). Zamiast po prostu skasować te wiersze, musimy zrozumieć **skąd się wzięły**.
Jeśli samolot nie ma odnotowanego czasu wylotu ani czasu w powietrzu (`air_time`), oznacza to, że **lot został odwołany (Cancelled Flight)**.

In [7]:
# 1. Oznaczenie lotów odwołanych
flights['is_cancelled'] = flights['dep_delay'].isna() | flights['arr_delay'].isna()
liczba_odwolanych = flights['is_cancelled'].sum()
print(f"Zidentyfikowano {liczba_odwolanych:,} odwołanych lotów w 2013 roku ({liczba_odwolanych/len(flights):.2%}).")

# 2. Pozotałe loty , które nie zostały odwałne
flights_clean = flights.dropna(subset=['dep_delay', 'arr_delay', 'air_time']).copy()

print(f"Liczba zrealizowanych lotów gotowych do analizy opóźnien: {len(flights_clean):,}")

Zidentyfikowano 9,430 odwołanych lotów w 2013 roku (2.80%).
Liczba zrealizowanych lotów gotowych do analizy opóźnien: 327,346


## Krok 5: Weryfikacja spójności fizycznej


In [9]:
# Sprawdzenie i odrzucenie wartości nielogicznych
przed_filtr = len(flights_clean)
flights_clean = flights_clean[(flights_clean['air_time'] > 0) & (flights_clean['distance'] > 0)]
po_filtr = len(flights_clean)

if przed_filtr != po_filtr:
    print(f"Usunięto {przed_filtr - po_filtr} wierszy z nielogicznym czasem lotu lub dystansem.")
else:
    print("Wszystkie czasy lotów i dystanse mają poprawne, dodatnie wartości.")

Wszystkie czasy lotów i dystanse mają poprawne, dodatnie wartości.


## Krok 6. Eksport danych

In [10]:
# Zapisanie gotowych plików do dalszej analizy
flights_clean.to_csv("data/flights_clean.csv", index=False)
weather_clean.to_csv("data/weather_clean.csv", index=False)

# dataset do lotów odloanych
flights.to_csv("data/flights_with_cancelled.csv", index=False)

print(" Sukces! Wyczyszczone dane zostały zapisane w folderze 'data/'.")

 Sukces! Wyczyszczone dane zostały zapisane w folderze 'data/'.
